# Ejemplo de Proyecto Final: Clasificador de Gatos vs Perros
Este notebook muestra un proyecto completo de visión por computador usando MobileNetV2 con transferencia de aprendizaje para clasificar imágenes de gatos y perros.

## 1. Descripción del proyecto
- Objetivo: clasificar imágenes como **gato** o **perro**.
- Dataset: subconjunto reducido del dataset de Kaggle Dogs vs Cats.
- Técnicas: Transfer Learning con MobileNetV2.

In [ ]:
print("Problema: Clasificación binaria entre gatos y perros usando imágenes.")

## 2. Preparación del conjunto de datos
- Dataset local con dos carpetas de clase dentro de `PetImages`: `Cat` y `Dog`.
- Se generará la partición de entrenamiento y validación automáticamente a partir de esas carpetas.


In [18]:
# Localizar el dataset en disco
from pathlib import Path

dataset_root = Path("vision/teoria/cats_and_dogs_filtered")
images_dir = dataset_root / "PetImages"

if images_dir.exists():
    print(f"Dataset encontrado en: {images_dir.resolve()}")
else:
    raise FileNotFoundError(
        "No se ha encontrado la carpeta cats_and_dogs_filtered/PetImages con las subcarpetas Cat y Dog."
    )


Dataset encontrado en: /home/jmsa/IESRafaelAlberti/RafaelAlberti25_26/Modulos/PIA/UD4/04-modelado-avanzado/vision/teoria/cats_and_dogs_filtered/PetImages


In [19]:
# Cargar imágenes válidas desde las carpetas Cat y Dog
import tensorflow as tf
from PIL import Image

img_size = (160, 160)
batch_size = 32
seed = 42

class_names = ["Cat", "Dog"]
class_dirs = [images_dir / class_name for class_name in class_names]
if not all(path.exists() for path in class_dirs):
    raise FileNotFoundError("Se esperaban las carpetas PetImages/Cat y PetImages/Dog.")

def is_valid_image(path: Path) -> bool:
    try:
        if path.stat().st_size == 0:
            return False
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

file_paths = []
labels = []
invalid_files = []

import numpy as np

valid_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for label, class_name in enumerate(class_names):
    for image_path in sorted(path for path in (images_dir / class_name).iterdir() if path.suffix.lower() in valid_suffixes):
        if is_valid_image(image_path):
            file_paths.append(str(image_path))
            labels.append(label)
        else:
            invalid_files.append(str(image_path))

if not file_paths:
    raise ValueError("No se encontraron imágenes válidas en PetImages.")

print(f"Imágenes válidas: {len(file_paths)}")
print(f"Imágenes descartadas: {len(invalid_files)}")
if invalid_files:
    print("Ejemplos descartados:", invalid_files[:5])

paths_ds = tf.data.Dataset.from_tensor_slices(file_paths)
labels_ds = tf.keras.utils.to_categorical(labels, num_classes=len(class_names))
labels_ds = tf.data.Dataset.from_tensor_slices(labels_ds)
dataset = tf.data.Dataset.zip((paths_ds, labels_ds))
dataset = dataset.shuffle(len(file_paths), seed=seed, reshuffle_each_iteration=False)

def load_image_with_pil(path):
    path = path.numpy().decode("utf-8")
    with Image.open(path) as img:
        image = img.convert("RGB")
        image = image.resize(img_size)
        image = np.asarray(image, dtype=np.float32)
    return image

def load_image(path, label):
    image = tf.py_function(load_image_with_pil, [path], Tout=tf.float32)
    image.set_shape((img_size[0], img_size[1], 3))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

train_size = int(0.8 * len(file_paths))
train_data = dataset.take(train_size).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_data = dataset.skip(train_size).batch(batch_size).prefetch(tf.data.AUTOTUNE)

print("Clases detectadas:", class_names)
print(f"Muestras entrenamiento: {train_size}")
print(f"Muestras validación: {len(file_paths) - train_size}")


Imágenes válidas: 24998
Imágenes descartadas: 2
Ejemplos descartados: ['vision/teoria/cats_and_dogs_filtered/PetImages/Cat/666.jpg', 'vision/teoria/cats_and_dogs_filtered/PetImages/Dog/11702.jpg']
Clases detectadas: ['Cat', 'Dog']
Muestras entrenamiento: 19998
Muestras validación: 5000


## 3. Entrenamiento del modelo (MobileNetV2)

In [20]:
base_model = tf.keras.applications.MobileNetV2(input_shape=img_size + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_data, validation_data=val_data, epochs=5)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9597 - loss: 0.1023

KeyboardInterrupt: 

## 4. Evaluación del modelo y resultados

In [21]:
import matplotlib.pyplot as plt

# Mostrar precisión y pérdida
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title("Precisión del modelo")
plt.xlabel("Épocas")
plt.ylabel("Precisión")
plt.show()

NameError: name 'history' is not defined

## 5. Conclusiones
- MobileNetV2 logra buena precisión con pocas épocas.
- Puede mejorarse con fine-tuning o aumento de datos.
- Proyecto sencillo pero aplicable a problemas reales.